# SEG QLoRA Small LLM Router

Fine-tune a small instruction model to route scientific search queries to exactly one label: `bm25`, `dense`, or `hybrid`.

Expected input files:
- `train_llm_router_data.jsonl`
- `test_llm_router_data.jsonl`

Expected output file:
- `test_llm_router_predictions.csv`


In [1]:
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes pandas scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 107.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 123.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 123.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3

In [2]:
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()
print(uploaded.keys())
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as zf:
            zf.extractall('.')
        print('extracted', name)
print([p.name for p in Path('.').glob('*llm_router_data.jsonl')])


KeyboardInterrupt: 

In [ ]:
import json
import pandas as pd
from datasets import Dataset

TRAIN_PATH = 'train_llm_router_data.jsonl'
TEST_PATH = 'test_llm_router_data.jsonl'

def read_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(TRAIN_PATH)
test_rows = read_jsonl(TEST_PATH)
print(len(train_rows), len(test_rows))
pd.Series([r['label'] for r in train_rows]).value_counts()


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_LENGTH = 384

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
LABELS = {'bm25', 'dense', 'hybrid'}

def format_example(row, include_answer=True):
    messages = [
        {'role': 'system', 'content': 'You are a query router for a scientific paper search engine. Choose exactly one retrieval route from: bm25, dense, hybrid.'},
        {'role': 'user', 'content': f"Academic query:\n{row['query']}\n\nRoute:"},
    ]
    if include_answer:
        messages.append({'role': 'assistant', 'content': row['label']})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=not include_answer)

def tokenize_example(row):
    text = format_example(row, include_answer=True)
    encoded = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding='max_length')
    encoded['labels'] = encoded['input_ids'].copy()
    return encoded

train_ds = Dataset.from_list(train_rows).map(tokenize_example, remove_columns=list(train_rows[0].keys()))
train_ds


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir='seg-qlora-router',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
trainer.train()


In [ ]:
import re
from tqdm.auto import tqdm

def normalize_label(text):
    text = text.strip().lower()
    for label in ['bm25', 'dense', 'hybrid']:
        if text.startswith(label) or re.search(rf'\b{label}\b', text):
            return label
    return 'hybrid'

@torch.no_grad()
def predict_label(row):
    prompt = format_example(row, include_answer=False)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_LENGTH).to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=4,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return normalize_label(generated), generated

pred_rows = []
for row in tqdm(test_rows):
    pred_label, raw_output = predict_label(row)
    pred_rows.append({
        'query_id': row['query_id'],
        'query': row['query'],
        'true_label': row.get('label'),
        'pred_label': pred_label,
        'raw_output': raw_output,
    })

pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv('test_llm_router_predictions.csv', index=False)
pred_df.head()


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print('accuracy', accuracy_score(pred_df['true_label'], pred_df['pred_label']))
print('macro_f1', f1_score(pred_df['true_label'], pred_df['pred_label'], average='macro'))
print(classification_report(pred_df['true_label'], pred_df['pred_label']))
files.download('test_llm_router_predictions.csv')
